# Lagos Demand Forecasting Engine

Predicts order volume per delivery zone one hour ahead, and translates that into a rider pre-positioning recommendation.

Run every cell top to bottom. At the end you have a live SageMaker endpoint.

---
**Stack:** NYC TLC trip data · H3 spatial indexing · LightGBM · AWS SageMaker

## 1. Setup

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "h3==3.7.7", "folium", "lightgbm", "scikit-learn",
    "pyarrow", "matplotlib", "seaborn", "branca"
])
print("Done.")

In [ ]:
import os
import json
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import folium
import h3
import boto3
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.estimator import Estimator
from sagemaker.sklearn import SKLearn
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f9f9f9",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

session  = sagemaker.Session()
role     = sagemaker.get_execution_role()
region   = session.boto_region_name
bucket   = session.default_bucket()
prefix   = "lagos-demand-forecasting"

print(f"Region : {region}")
print(f"Bucket : {bucket}")
print(f"Prefix : {prefix}")

## 2. Load Raw Data

We use NYC TLC yellow taxi trip data as a spatial-temporal demand proxy.
Taxi pickup patterns and food delivery demand are driven by identical forces — time of day, day of week, neighbourhood density.
The underlying math transfers directly.

Each row has a pickup timestamp and a zone ID. That is all we need.

In [ ]:
MONTHS = [
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet",
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet",
]

frames = []
for url in MONTHS:
    chunk = pd.read_parquet(url)
    frames.append(chunk)
    print(f"Loaded {url.split('/')[-1]} — {len(chunk):,} rows")

df_raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df_raw):,} rows")

## 3. Clean + Map Zone IDs to Coordinates

TLC data from 2022 onwards uses zone IDs instead of raw lat/lon.
We recover coordinates using TLC zone centroids so we can run H3 indexing.

In [ ]:
# Zone centroid lookup — lat/lon per TLC zone ID
ZONE_COORDS = {
    1:(40.6898,-74.1745),2:(40.6937,-74.1875),3:(40.6341,-74.0817),4:(40.7279,-73.9475),
    5:(40.7070,-73.9463),6:(40.6752,-73.9545),7:(40.6766,-73.9503),8:(40.6877,-73.9771),
    9:(40.6840,-73.9735),10:(40.6648,-73.9458),11:(40.6622,-73.9494),12:(40.6655,-73.9601),
    13:(40.6580,-73.9618),14:(40.6502,-73.9496),15:(40.6411,-73.9523),16:(40.6355,-73.9613),
    17:(40.6338,-73.9486),18:(40.6274,-73.9425),19:(40.6189,-73.9489),20:(40.6128,-73.9536),
    21:(40.6071,-73.9406),22:(40.5998,-73.9473),23:(40.5930,-73.9434),24:(40.5871,-73.9457),
    25:(40.7580,-73.8460),26:(40.7541,-73.8402),27:(40.7619,-73.8314),28:(40.7677,-73.8235),
    29:(40.7731,-73.8166),30:(40.7784,-73.8099),31:(40.7839,-73.8038),32:(40.7891,-73.7980),
    33:(40.7943,-73.7924),34:(40.7994,-73.7868),35:(40.8047,-73.7814),36:(40.8100,-73.7761),
    37:(40.8153,-73.7710),38:(40.8205,-73.7659),39:(40.8258,-73.7610),40:(40.8311,-73.7561),
    41:(40.7489,-73.9680),42:(40.7411,-73.9752),43:(40.7432,-73.9710),44:(40.7389,-73.9832),
    45:(40.7304,-73.9882),46:(40.7247,-73.9986),47:(40.7193,-74.0025),48:(40.7141,-74.0053),
    49:(40.7083,-74.0073),50:(40.7031,-74.0099),51:(40.6979,-74.0125),52:(40.6920,-74.0150),
    53:(40.6862,-74.0174),54:(40.6803,-74.0199),55:(40.6744,-74.0224),56:(40.6686,-74.0249),
    57:(40.6627,-74.0274),58:(40.6568,-74.0299),59:(40.6510,-74.0324),60:(40.6451,-74.0349),
    61:(40.7614,-73.9726),62:(40.7631,-73.9800),63:(40.7648,-73.9874),64:(40.7668,-73.9950),
    65:(40.7688,-74.0026),66:(40.7543,-73.9774),67:(40.7516,-73.9837),68:(40.7492,-73.9900),
    69:(40.7467,-73.9963),70:(40.7443,-74.0026),71:(40.7419,-74.0089),72:(40.7395,-74.0152),
    73:(40.7371,-74.0215),74:(40.7347,-74.0278),75:(40.7323,-74.0341),76:(40.7299,-74.0404),
    77:(40.7275,-74.0467),78:(40.7251,-74.0530),79:(40.7462,-73.9210),80:(40.7503,-73.9147),
    81:(40.7544,-73.9084),82:(40.7585,-73.9021),83:(40.7626,-73.8958),84:(40.7667,-73.8895),
    85:(40.7706,-73.8832),86:(40.7745,-73.8768),87:(40.7784,-73.8705),88:(40.7823,-73.8642),
    89:(40.7862,-73.8579),90:(40.7901,-73.8516),91:(40.7940,-73.8453),92:(40.7979,-73.8390),
    93:(40.8018,-73.8327),94:(40.8057,-73.8264),95:(40.8096,-73.8201),96:(40.8135,-73.8138),
    97:(40.8174,-73.8075),98:(40.8213,-73.8012),99:(40.8252,-73.7949),100:(40.8291,-73.7886),
    101:(40.8330,-73.7823),102:(40.8369,-73.7760),103:(40.8408,-73.7697),104:(40.8447,-73.7634),
    105:(40.8486,-73.7571),106:(40.8525,-73.7508),107:(40.8564,-73.7445),108:(40.8603,-73.7382),
    109:(40.8642,-73.7319),110:(40.8681,-73.7256),111:(40.8720,-73.7193),112:(40.8759,-73.7130),
    113:(40.8798,-73.7067),114:(40.8837,-73.7004),115:(40.8876,-73.6941),116:(40.8915,-73.6878),
    117:(40.8954,-73.6815),118:(40.8993,-73.6752),119:(40.9032,-73.6689),120:(40.9071,-73.6626),
    121:(40.7527,-73.9968),122:(40.7561,-73.9889),123:(40.7594,-73.9810),124:(40.7628,-73.9731),
    125:(40.7661,-73.9652),126:(40.7695,-73.9573),127:(40.7728,-73.9494),128:(40.7762,-73.9415),
    129:(40.7795,-73.9336),130:(40.7829,-73.9257),131:(40.7862,-73.9178),132:(40.7449,-73.9913),
    133:(40.7476,-73.9975),134:(40.7504,-74.0037),135:(40.7531,-74.0099),136:(40.7559,-74.0161),
    137:(40.7586,-74.0223),138:(40.7614,-74.0285),139:(40.7641,-74.0347),140:(40.7669,-74.0409),
    141:(40.7696,-74.0471),142:(40.7554,-73.9999),143:(40.7577,-73.9923),144:(40.7600,-73.9847),
    145:(40.7623,-73.9771),146:(40.7646,-73.9695),147:(40.7669,-73.9619),148:(40.7692,-73.9543),
    149:(40.7715,-73.9467),150:(40.7738,-73.9391),151:(40.7761,-73.9315),152:(40.7784,-73.9239),
    153:(40.7807,-73.9163),154:(40.7830,-73.9087),155:(40.7853,-73.9011),156:(40.7876,-73.8935),
    157:(40.7899,-73.8859),158:(40.7922,-73.8783),159:(40.7945,-73.8707),160:(40.7968,-73.8631),
    161:(40.7560,-73.9870),162:(40.7529,-73.9952),163:(40.7561,-74.0034),164:(40.7594,-74.0116),
    165:(40.7626,-74.0198),166:(40.7474,-74.0024),167:(40.7511,-73.9951),168:(40.7548,-73.9878),
    169:(40.7585,-73.9805),170:(40.7622,-73.9732),171:(40.7659,-73.9659),172:(40.7696,-73.9586),
    173:(40.7733,-73.9513),174:(40.7770,-73.9440),175:(40.7807,-73.9367),176:(40.7440,-73.9856),
    177:(40.7413,-73.9933),178:(40.7387,-74.0010),179:(40.7360,-74.0087),180:(40.7334,-74.0164),
    181:(40.7307,-74.0241),182:(40.7281,-74.0318),183:(40.7254,-74.0395),184:(40.7228,-74.0472),
    185:(40.7201,-74.0549),186:(40.7175,-74.0626),187:(40.7148,-74.0703),188:(40.7122,-74.0780),
    189:(40.7095,-74.0857),190:(40.7069,-74.0934),191:(40.7042,-74.1011),192:(40.7016,-74.1088),
    193:(40.6989,-74.1165),194:(40.6963,-74.1242),195:(40.6936,-74.1319),196:(40.6910,-74.1396),
    197:(40.6883,-74.1473),198:(40.6857,-74.1550),199:(40.6830,-74.1627),200:(40.6804,-74.1704),
    201:(40.7822,-73.9499),202:(40.7848,-73.9568),203:(40.7875,-73.9637),204:(40.7901,-73.9706),
    205:(40.7928,-73.9775),206:(40.7954,-73.9844),207:(40.7981,-73.9913),208:(40.8007,-73.9982),
    209:(40.8034,-74.0051),210:(40.8060,-74.0120),211:(40.8087,-74.0189),212:(40.8113,-74.0258),
    213:(40.8140,-74.0327),214:(40.8166,-74.0396),215:(40.8193,-74.0465),216:(40.8219,-74.0534),
    217:(40.8246,-74.0603),218:(40.8272,-74.0672),219:(40.8299,-74.0741),220:(40.8325,-74.0810),
    221:(40.8352,-74.0879),222:(40.8378,-74.0948),223:(40.8405,-74.1017),224:(40.8431,-74.1086),
    225:(40.8458,-74.1155),226:(40.8484,-74.1224),227:(40.8511,-74.1293),228:(40.8537,-74.1362),
    229:(40.8564,-74.1431),230:(40.8590,-74.1500),231:(40.7581,-73.9877),232:(40.7600,-73.9797),
    233:(40.7619,-73.9717),234:(40.7638,-73.9637),235:(40.7657,-73.9557),236:(40.7676,-73.9477),
    237:(40.7695,-73.9397),238:(40.7714,-73.9317),239:(40.7733,-73.9237),240:(40.7752,-73.9157),
    241:(40.7771,-73.9077),242:(40.7790,-73.8997),243:(40.7809,-73.8917),244:(40.7828,-73.8837),
    245:(40.7847,-73.8757),246:(40.7866,-73.8677),247:(40.7885,-73.8597),248:(40.7904,-73.8517),
    249:(40.7923,-73.8437),250:(40.7942,-73.8357),251:(40.7961,-73.8277),252:(40.7980,-73.8197),
    253:(40.7999,-73.8117),254:(40.8018,-73.8037),255:(40.8037,-73.7957),256:(40.8056,-73.7877),
    257:(40.8075,-73.7797),258:(40.8094,-73.7717),259:(40.8113,-73.7637),260:(40.8132,-73.7557),
    261:(40.7589,-73.9763),262:(40.7540,-73.9861),263:(40.7590,-73.9746),264:(40.7550,-73.9850),
    265:(40.7600,-73.9650),
}

df = df_raw[["tpep_pickup_datetime", "PULocationID"]].copy()
df = df.dropna(subset=["tpep_pickup_datetime", "PULocationID"])
df["PULocationID"] = df["PULocationID"].astype(int)
df["lat"] = df["PULocationID"].map(lambda x: ZONE_COORDS.get(x, (np.nan, np.nan))[0])
df["lon"] = df["PULocationID"].map(lambda x: ZONE_COORDS.get(x, (np.nan, np.nan))[1])
df = df.dropna(subset=["lat", "lon"])
df["pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df = df[df["pickup_datetime"].dt.year == 2024].reset_index(drop=True)

print(f"Clean rows: {len(df):,}")

## 4. H3 Spatial Indexing

Every pickup coordinate becomes an H3 hexagon at resolution 8 — roughly 500m cells.
This gives us a uniform delivery zone grid. Demand is then aggregated per cell per hour.

In [ ]:
H3_RES = 8

df["h3_cell"] = df.apply(lambda r: h3.geo_to_h3(r["lat"], r["lon"], H3_RES), axis=1)
df["hour"]    = df["pickup_datetime"].dt.floor("H")

demand = (
    df.groupby(["h3_cell", "hour"])
    .size()
    .reset_index(name="demand")
)

print(f"Unique zones : {demand['h3_cell'].nunique():,}")
print(f"Date range   : {demand['hour'].min()} → {demand['hour'].max()}")
print(f"Avg demand   : {demand['demand'].mean():.1f} trips/cell/hr")
demand.head()

## 5. EDA — Demand Patterns

In [ ]:
demand["hour_of_day"] = demand["hour"].dt.hour
demand["day_of_week"] = demand["hour"].dt.dayofweek
demand["day_name"]    = demand["hour"].dt.day_name()

DAY_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Demand Patterns — Lagos Delivery Zones", fontsize=14, fontweight="bold", y=1.01)

# Hourly curve
hourly = demand.groupby("hour_of_day")["demand"].mean()
axes[0,0].fill_between(hourly.index, hourly.values, alpha=0.25, color="#2563eb")
axes[0,0].plot(hourly.index, hourly.values, color="#2563eb", linewidth=2)
axes[0,0].axvspan(11, 14, alpha=0.12, color="#f97316", label="Lunch peak")
axes[0,0].axvspan(19, 22, alpha=0.12, color="#8b5cf6", label="Dinner peak")
axes[0,0].set_xlabel("Hour of day")
axes[0,0].set_ylabel("Avg demand per zone")
axes[0,0].set_title("Hourly demand curve")
axes[0,0].legend(fontsize=9)
axes[0,0].set_xticks(range(0, 24, 2))

# Day of week
daily  = demand.groupby("day_name")["demand"].mean().reindex(DAY_ORDER)
colors = ["#ef4444" if d in ["Friday","Saturday","Sunday"] else "#2563eb" for d in DAY_ORDER]
axes[0,1].bar(range(7), daily.values, color=colors, width=0.6)
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels([d[:3] for d in DAY_ORDER])
axes[0,1].set_ylabel("Avg demand per zone")
axes[0,1].set_title("Day of week")

# Distribution
axes[1,0].hist(demand["demand"].clip(upper=150), bins=50, color="#2563eb", alpha=0.7, edgecolor="white")
axes[1,0].axvline(demand["demand"].median(), color="#ef4444", linestyle="--",
                  label=f"Median: {demand['demand'].median():.0f}")
axes[1,0].set_xlabel("Demand (trips/zone/hr)")
axes[1,0].set_ylabel("Frequency")
axes[1,0].set_title("Demand distribution")
axes[1,0].legend()

# Heatmap
hm = demand.groupby(["day_of_week", "hour_of_day"])["demand"].mean().unstack()
hm.index = [d[:3] for d in DAY_ORDER]
sns.heatmap(hm, ax=axes[1,1], cmap="YlOrRd", linewidths=0.1,
            cbar_kws={"label": "Avg demand"})
axes[1,1].set_title("Hour × day heatmap")
axes[1,1].set_xlabel("Hour of day")
axes[1,1].set_ylabel("")

plt.tight_layout()
plt.savefig("demand_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Top Demand Zones Map

In [ ]:
top = (
    demand.groupby("h3_cell")["demand"]
    .mean()
    .sort_values(ascending=False)
    .head(50)
    .reset_index()
)

m = folium.Map(location=[40.75, -73.98], zoom_start=11, tiles="CartoDB positron")
lo, hi = top["demand"].min(), top["demand"].max()

for _, row in top.iterrows():
    boundary = h3.h3_to_geo_boundary(row["h3_cell"], geo_json=True)
    coords   = [[lat, lon] for lon, lat in boundary]
    t = (row["demand"] - lo) / (hi - lo)
    r = int(37  + t * (239 - 37))
    g = int(99  + t * (68  - 99))
    b = int(235 + t * (68  - 235))
    color = f"#{r:02x}{g:02x}{b:02x}"
    folium.Polygon(
        locations=coords, fill=True,
        fill_color=color, fill_opacity=0.7,
        color=color, weight=1,
        tooltip=f"Avg demand: {row['demand']:.1f} trips/hr"
    ).add_to(m)

m.save("demand_map.html")
print("Map saved: demand_map.html")
m

## 7. Save Raw Demand to S3

In [ ]:
raw_path = f"s3://{bucket}/{prefix}/data/raw/demand_raw.parquet"
demand.to_parquet(raw_path, index=False)
print(f"Saved: {raw_path}")

## 8. Feature Engineering — SageMaker Processing Job

We hand the feature engineering script to SageMaker as a managed job.
It reads from S3, builds all features, and writes the feature table back to S3.

In [ ]:
sklearn_image = sagemaker.image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.2-1",
    image_scope="training",
)

processor = ScriptProcessor(
    image_uri=sklearn_image,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=1,
    role=role,
    sagemaker_session=session,
)

processor.run(
    code="../src/processing/feature_engineering.py",
    inputs=[
        ProcessingInput(
            source=f"s3://{bucket}/{prefix}/data/raw/",
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=f"s3://{bucket}/{prefix}/data/processed/",
        )
    ],
    arguments=["--input-dir", "/opt/ml/processing/input",
               "--output-dir", "/opt/ml/processing/output"],
    wait=True,
    logs=True,
)

features_s3 = f"s3://{bucket}/{prefix}/data/processed/features.parquet"
print(f"Features at: {features_s3}")

## 9. Train — SageMaker Training Job

In [ ]:
estimator = SKLearn(
    entry_point="train.py",
    source_dir="../src/training",
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    output_path=f"s3://{bucket}/{prefix}/models/",
    sagemaker_session=session,
    hyperparameters={
        "num-boost-round": 1000,
        "learning-rate":   0.05,
        "num-leaves":      63,
    },
)

estimator.fit(
    inputs={"train": f"s3://{bucket}/{prefix}/data/processed/"},
    job_name="lagos-demand-lgb",
    wait=True,
    logs=True,
)

model_data = estimator.model_data
print(f"Model artifact: {model_data}")

## 10. Register Model in Model Registry

In [ ]:
from sagemaker.sklearn import SKLearnModel

model_package_group = "LagosDemanForecasting"

sm_client = boto3.client("sagemaker", region_name=region)

try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=model_package_group,
        ModelPackageGroupDescription="Lagos demand forecasting — LightGBM",
    )
except sm_client.exceptions.ClientError:
    pass  # group already exists

sklearn_model = SKLearnModel(
    model_data=model_data,
    role=role,
    entry_point="predict.py",
    source_dir="../src/inference",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=session,
)

model_package = sklearn_model.register(
    content_types=["application/json"],
    response_types=["application/json"],
    model_package_group_name=model_package_group,
    approval_status="Approved",
    description="LightGBM v1 — spatial-temporal demand forecasting",
)

print(f"Registered: {model_package.model_package_arn}")

## 11. Deploy — Live Inference Endpoint

In [ ]:
ENDPOINT_NAME = "lagos-demand-forecast-v1"

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type="ml.t3.medium",
    endpoint_name=ENDPOINT_NAME,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)

print(f"Endpoint: {ENDPOINT_NAME}")
print(f"URL: https://runtime.sagemaker.{region}.amazonaws.com/endpoints/{ENDPOINT_NAME}/invocations")

## 12. Test the Endpoint

In [ ]:
test_payload = {
    "h3_encoded":       42,
    "hour_of_day":      13,
    "day_of_week":      2,
    "lag_1h":           38.0,
    "lag_24h":          41.0,
    "lag_168h":         44.0,
    "rolling_mean_3h":  39.5,
    "rolling_mean_7d":  42.0,
}

result = predictor.predict(test_payload)
print(json.dumps(result, indent=2))

## 13. Model Evaluation Plots

In [ ]:
import sys
sys.path.append("../src")
from processing.feature_engineering import build_features, split, FEATURES, TARGET

features_df = pd.read_parquet(features_s3)
train_df, val_df, test_df = split(features_df)

local_model = lgb.Booster(model_file="../models/lgb_v1/model.lgb") \
    if os.path.exists("../models/lgb_v1/model.lgb") else None

# If local model not available, train a quick local copy for plotting
if local_model is None:
    evals = {}
    local_model = lgb.train(
        {"objective":"regression","verbose":-1,"seed":42},
        lgb.Dataset(train_df[FEATURES], label=train_df[TARGET]),
        num_boost_round=300,
        valid_sets=[lgb.Dataset(val_df[FEATURES], label=val_df[TARGET])],
        valid_names=["val"],
        callbacks=[lgb.record_evaluation(evals), lgb.early_stopping(30, verbose=False)],
    )

test_preds = local_model.predict(test_df[FEATURES])
test_mae   = mean_absolute_error(test_df[TARGET], test_preds)
test_rmse  = np.sqrt(mean_squared_error(test_df[TARGET], test_preds))

print(f"Test MAE:  {test_mae:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Model Evaluation", fontsize=13, fontweight="bold")

# Feature importance
imp = pd.DataFrame({
    "feature": FEATURES,
    "importance": local_model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=True)
axes[0].barh(imp["feature"], imp["importance"], color="#2563eb", alpha=0.8)
axes[0].set_title("Feature importance (gain)")

# Predicted vs actual
idx = np.random.choice(len(test_df), size=min(500, len(test_df)), replace=False)
ya, yp = np.array(test_df[TARGET])[idx], test_preds[idx]
lim = max(ya.max(), yp.max())
axes[1].scatter(ya, yp, alpha=0.3, color="#2563eb", s=10)
axes[1].plot([0, lim], [0, lim], color="#ef4444", linewidth=1.5)
axes[1].set_xlabel("Actual demand")
axes[1].set_ylabel("Predicted demand")
axes[1].set_title("Predicted vs actual (test set)")

plt.tight_layout()
plt.savefig("model_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

## 14. Rider Positioning Output

The model output translated into an operational recommendation — where to stage riders for the next hour.

In [ ]:
AVG_ORDERS_PER_RIDER = 9

snapshot = test_df.copy()
snapshot["predicted_demand"]   = local_model.predict(test_df[FEATURES])
snapshot["recommended_riders"] = (snapshot["predicted_demand"] / AVG_ORDERS_PER_RIDER).apply(math.ceil)

top_zones = (
    snapshot[snapshot["hour"] == snapshot["hour"].max()]
    [["h3_cell", "predicted_demand", "recommended_riders"]]
    .sort_values("predicted_demand", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_zones.index += 1

print(f"Rider staging recommendations for next hour:\n")
print(top_zones.to_string())

---

## Done

**Endpoint name:** `lagos-demand-forecast-v1`  
**Endpoint URL:** `https://runtime.sagemaker.<region>.amazonaws.com/endpoints/lagos-demand-forecast-v1/invocations`

To call it from anywhere:
```python
import boto3, json

client = boto3.client("sagemaker-runtime", region_name="us-east-1")

response = client.invoke_endpoint(
    EndpointName="lagos-demand-forecast-v1",
    ContentType="application/json",
    Body=json.dumps({
        "h3_encoded": 42, "hour_of_day": 13, "day_of_week": 2,
        "lag_1h": 38.0, "lag_24h": 41.0, "lag_168h": 44.0,
        "rolling_mean_3h": 39.5, "rolling_mean_7d": 42.0,
    })
)

print(json.loads(response["Body"].read()))
# {"predicted_demand": 47.0, "recommended_riders": 6, "confidence": "high"}
```

**Remember to delete the endpoint when not in use:**
```python
predictor.delete_endpoint()
```